In [ ]:
# CELL 1 - SETUP: mount Drive, get the code, install it, show the GPU
import importlib, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<YOUR_USER>/<YOUR_REPO>.git"   # TODO student: your GitHub repository
BRANCH = "<BRANCH_OR_TAG>"                                     # TODO student: branch / tag / commit to run
CODE_DATASET = "/content/drive/MyDrive/genai-a1/code"                           # fallback: code zip uploaded as a private dataset
DRIVE_ROOT = "/content/drive/MyDrive/genai-a1"                 # persistent folder (checkpoints, Optuna DB, data)

from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir("/content/data"):                         # data lives on local disk during training
    shutil.copytree(f"{DRIVE_ROOT}/data", "/content/data")
if not os.path.isdir("repo"):
    if subprocess.call(["git", "clone", "--branch", BRANCH, REPO_URL, "repo"]) != 0:
        print("git clone failed -> copying the code from Drive instead")
        shutil.copytree(CODE_DATASET, "repo")
os.chdir("repo")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "-r", "requirements.txt"])
importlib.invalidate_caches()
subprocess.call(["nvidia-smi"])


In [ ]:
# CELL 2 - PARAMETERS: the only cell you normally edit
TASK = "t1"                                  # t1 | t2cls | t2spec | t3 | t4
MODE = "train"                               # "train" (run_training), "tune" (run_study), "benchmark" (time the GPU, first thing to run)
                                             # or "final_config" (t2spec only: write configs/task2_specialist_final.yaml from the finished study)
CONFIG = "configs/task1_universal.yaml"      # task config file
DEVICE_PROFILE = "colab"                 # picks configs/devices/colab.yaml
RESUME = "auto"                              # "auto" = newest ckpt_last.pt in the persistent folder; None = fresh run; or a path
OVERRIDES = {}                              # optional config overrides, e.g. {"train.max_steps": 50, "run.smoke": True}
# Task 2: TASK = "t2cls" (classifier) or "t2spec" (specialists). Each specialist is a separate run, choose it with
#   OVERRIDES = {"specialist.corruption": "salt"}     # or "blur" / "occlusion"; the shared study ignores it
# Specialist finals use the study result:  CONFIG = "configs/task2_specialist_final.yaml"  (after MODE = "final_config")
# NOTE: OVERRIDES values are parsed as YAML. To set a value to None write the string "null", not None
#       (None would be turned into the text "None").


In [ ]:
# CELL 3 - RUN: call the repository code (no logic lives in this notebook)
import shutil
from pathlib import Path
import subprocess, sys
from genai.tasks.cli import build_config, entry_point

cfg = build_config(TASK, CONFIG, DEVICE_PROFILE, [f"{k}={v}" for k, v in OVERRIDES.items()])

def on_checkpoint(path):
    """Copy every saved checkpoint / Optuna DB (and the run's log files) to Drive, same folder layout."""
    path, out_root, persist = Path(path), Path(cfg["output_root"]), Path(cfg["persist_root"])
    for name in (path.name, "metrics.jsonl", "config.yaml"):
        src = path.parent / name
        if src.exists():
            dest = persist / src.relative_to(out_root)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)

if MODE == "benchmark":
    # times a few batch sizes with/without AMP and appends rows to docs/BENCHMARKS.md (copy those rows back to your PC)
    subprocess.check_call([sys.executable, "scripts/benchmark.py", "--model", TASK, "--batch", "64", "128", "256", "512",
                           "--device-profile", DEVICE_PROFILE])
    result = "docs/BENCHMARKS.md"
elif MODE == "final_config":
    # Task 2 specialists: copy the best hyperparameters of the shared study into a full training config
    from genai.tasks.task2.specialist import write_final_config
    result = write_final_config(f"{cfg['output_root']}/optuna/t2_specialist_shared.db", "configs/task2_specialist_final.yaml")
elif MODE == "train":
    result = entry_point(TASK, "train", "run_training")(cfg, resume=RESUME, on_checkpoint=on_checkpoint)
else:
    result = entry_point(TASK, "tune", "run_study")(cfg, on_checkpoint=on_checkpoint)
print("finished:", result)


In [ ]:
# CELL 4 - OUTPUTS: everything was copied to Drive by on_checkpoint; list what is there to download
from pathlib import Path
persist = Path(cfg["persist_root"])
for pattern in ("runs/*/*/ckpt_best.pt", "runs/*/*/ckpt_last.pt", "runs/*/*/metrics.jsonl", "runs/*/*/samples/*.png", "optuna/*.db"):
    for p in sorted(persist.glob(pattern)):
        print(f"{p.stat().st_size / 2**20:8.1f} MB  {p}")
print("Download from Drive and put the run folder under artifacts/runs/ and the .db under artifacts/optuna/ on your PC.")
